# M3 · Less computation: sampling steps and Amdahl's law

**Question:** Why is changing 40 steps to 8 steps not an exactly 5× end-to-end gain? Recompute B0/L1 speedup, then derive the bound with the measured DiT fraction. Turbo changes the model/schedule and must pass a separate quality review.

**Learning loop:** predict → inspect frozen evidence → compute → critique → propose a controlled experiment.

Reference lesson: [open course/M03_less_computation/LESSON.md](../course/M03_less_computation/LESSON.md). Switch language: [中文](./03_less_computation.zh-CN.ipynb).

**Default mode:** CPU-only, internet-free, read-only. GPU work is never run implicitly.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root (see notebooks/README).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from lab import notebook_support as ns
import statistics
print("Course repository:", ROOT)
print("Mode: OFFLINE / committed reference evidence; NO GPU work is launched by default.")


## Inspect and recompute real evidence

Run cells in order. Explain every denominator and scope.

In [ ]:
rows = ns.summary_rows()
b, l = rows["B0"], rows["L1"]
observed = b["warm_total_median_s"] / l["warm_total_median_s"]
p = b["warm_ksampler_median_s"] / b["warm_total_median_s"]
upper = 1 / ((1 - p) + p / 5)
print(f"B0={b['warm_total_median_s']:.3f}s, L1={l['warm_total_median_s']:.3f}s")
print(f"Measured speedup={observed:.3f}x; Amdahl estimate at 5x denoiser={upper:.3f}x")
assert 4 < observed < 5
print("These are distinct model/schedule variants; the images are NOT lossless parity.")
plot_values = {"B0": b["warm_total_median_s"], "Turbo L1": l["warm_total_median_s"]}
plot_ylabel, plot_title = "warm seconds", "Reduce sampling work (not a lossless edit)"


In [ ]:
SHOW_PLOT = False  # Optional: requires matplotlib in the notebook kernel
if SHOW_PLOT:
    import matplotlib.pyplot as plt
    labels = list(plot_values)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(labels, [plot_values[label] for label in labels])
    ax.set_ylabel(plot_ylabel)
    ax.set_title(plot_title)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("Optional plot skipped; set SHOW_PLOT=True to visualize the measured values.")


In [ ]:
# Student-editable counterfactual: same measured denoiser fraction, different local speedups.
for local_accel in (2, 5, 10, 100):
    bound = 1 / ((1 - p) + p / local_accel)
    print(f"Local denoiser {local_accel:>3}x -> ideal E2E bound {bound:.3f}x")
print("Amdahl modeling is hypothetical; actual Turbo changes model/schedule and needs quality review.")


## Optional physical GPU smoke — explicit opt-in

Requires the staged ROCm environment and sufficient model/GPU resources. Never used in CPU CI. The run goes to `runs/`, not official evidence.

In [ ]:
RUN_GPU = False  # Only switch on after checking platform quota + available models
if RUN_GPU:
    # Also set QIR_NOTEBOOK_GPU_ACK=YES in your own kernel environment.
    # The helper enforces --smoke and writes under runs/, never evidence/.
    ns.gpu_smoke("L1", "notebook-l1", confirmed=True)
else:
    print("GPU NOT_RUN; evidence analysis above is valid without GPU access.")


## Your turn / Self-check

Change the hypothetical denoiser speedup from 5× to 2× and 10× in the equation. What becomes the bottleneck? Why can a schedule change improve speed but alter image composition?

**Evidence discipline:** Answer in your own words. This notebook's numbers come from existing committed records, not your GPU. If your optional smoke runs, its data must remain under `runs/` and may not be presented as a full-protocol median.
